In [ ]:
user_item_sparse = lil_matrix((n_users, n_items), dtype=np.int8)
    
    # Заполняем матрицу
for _, row in tqdm(df_completed.iterrows(), total=len(df_completed)):
    user_idx = user_to_idx[row['user_id']]
    item_idx = item_to_idx[row['item_id']]
    user_item_sparse[user_idx, item_idx] = 1

In [ ]:
def fastest_itemknn(target_user_id, watched_items, df_interactions, n_similar_users=10):
    """Самый быстрый вариант - используем готовые агрегации"""
    
    # Находим пользователей, которые смотрели хотя бы один из тех же фильмов
    potential_similar_users = df_interactions[
        (df_interactions['item_id'].isin(watched_items)) & 
        (df_interactions['user_id'] != target_user_id) &
        (df_interactions['completed'] == 1)
    ]['user_id'].unique()
    
    # Для каждого потенциально похожего пользователя считаем схожесть
    user_similarity = {}
    
    for user_id in potential_similar_users:
        # Фильмы, которые смотрел этот пользователь
        user_watched = set(df_interactions[
            (df_interactions['user_id'] == user_id) & 
            (df_interactions['completed'] == 1)
        ]['item_id'])
        
        # Схожесть по количеству общих фильмов (упрощенная метрика)
        common_items = len(set(watched_items) & user_watched)
        similarity = common_items / len(set(watched_items) | user_watched) if len(set(watched_items) | user_watched) > 0 else 0
        
        if similarity > 0:
            user_similarity[user_id] = similarity
    
    # Сортируем по убыванию схожести
    similar_users = sorted(user_similarity.items(), key=lambda x: x[1], reverse=True)[:n_similar_users]
    
    return similar_users

# Самый быстрый вариант
print("Запускаем быстрый вариант...")
similar_users_fast = fastest_itemknn(random_user_id, watched_items, df_interactions_train)

print(f"Найдено {len(similar_users_fast)} похожих пользователей:")
for i, (user_id, score) in enumerate(similar_users_fast, 1):
    print(f"{i}. User {user_id} (схожесть: {score:.4f})")

In [ ]:
# df_completed = df_interactions_train[df_interactions_train['completed'] == 1]
    
#     # Создаем разреженную матрицу пользователь-фильм
# user_item = df_completed.pivot_table(
#     index='user_id', 
#     columns='item_id', 
#     values='completed', 
#     aggfunc='count', 
#     fill_value=0
#     )

In [ ]:
# def find_similar_users_simple(target_user_id, watched_items, df_interactions, min_common_items=2):
#     """
#     Находит пользователей, которые смотрели те же фильмы, что и целевой пользователь
#     """
#     # Находим всех пользователей, которые смотрели хотя бы один из фильмов целевого пользователя
#     similar_users_candidates = df_interactions[
#         df_interactions['item_id'].isin(watched_items) & 
#         (df_interactions['user_id'] != target_user_id)
#     ]
    
#     # Группируем по пользователям и считаем количество общих фильмов
#     user_similarity = similar_users_candidates.groupby('user_id').agg({
#         'item_id': lambda x: len(set(x) & set(watched_items)),  # Количество общих фильмов
#         'completed': 'mean'  # Средний процент завершения для оценки "похожести вкуса"
#     }).reset_index()
    
#     user_similarity.columns = ['user_id', 'common_items_count', 'avg_completion_rate']
    
#     # Фильтруем по минимальному количеству общих фильмов
#     similar_users = user_similarity[user_similarity['common_items_count'] >= min_common_items]
    
#     # Сортируем по количеству общих фильмов (по убыванию)
#     similar_users = similar_users.sort_values('common_items_count', ascending=False)
    
#     return similar_users

# # Применяем функцию
# similar_users = find_similar_users_simple(random_user_id, watched_items, df_interactions_train)
# print(f"Найдено похожих пользователей: {len(similar_users)}")
# print(similar_users.head(10))

In [ ]:
def recommend_items_simple(target_user_id, watched_items, similar_users, df_interactions, df_items, top_n=10):
    """
    Простой подход: самые популярные фильмы среди похожих пользователей
    """
    # Берем топ-10 самых похожих пользователей
    top_similar_users = similar_users.head(10)['user_id'].tolist()
    
    # Фильмы, которые смотрели эти пользователи
    recommendations = df_interactions[
        df_interactions['user_id'].isin(top_similar_users) &
        ~df_interactions['item_id'].isin(watched_items)
    ]
    
    # Считаем популярность
    item_popularity = recommendations['item_id'].value_counts().reset_index()
    item_popularity.columns = ['item_id', 'view_count']
    
    # Добавляем информацию о фильмах
    final_recommendations = item_popularity.merge(
        df_items[['item_id', 'title', 'genres', 'release_year']], 
        on='item_id', 
        how='left'
    )
    
    return final_recommendations.head(top_n)

# Простые рекомендации
simple_recommendations = recommend_items_simple(
    random_user_id, watched_items, similar_users, df_interactions_train, df_items
)
print("\nПростые рекомендации (самые популярные у похожих пользователей):")
print(simple_recommendations[['item_id', 'title', 'genres', 'release_year', 'view_count']])

In [ ]:
# users_watched_same_items = df_interactions_train[
#     df_interactions['item_id'].isin(watched_items)]['user_id'].unique()

In [ ]:
# relevant_users = np.append(users_watched_same_items, random_user_id)
# user_item_matrix = df_interactions[
#     df_interactions['user_id'].isin(relevant_users)].pivot_table(
#     index='user_id', 
#     columns='item_id', 
#     values='completed',  # Используем completed как рейтинг
#     fill_value=0
# )

In [ ]:
# def find_similar_users(user_id, watched_items, df_interactions_train, n_similar=10):
#     # Пользователи, которые смотрели те же фильмы
#     similar_users = df_interactions_train[
#         df_interactions_train['item_id'].isin(watched_items) & 
#         (df_interactions_train['user_id'] != user_id)
#     ]['user_id'].value_counts().head(n_similar)
    
#     return similar_users.index.tolist()

In [ ]:
# def get_recommendations(similar_users, df_interactions_train, n_recommend=5):
#     # Фильмы, популярные у похожих пользователей
#     recommendations = df_interactions_train[
#         df_interactions_train['user_id'].isin(similar_users)
#     ]['item_id'].value_counts().head(n_recommend)
    
#     return recommendations.index.tolist()

In [ ]:
def create_user_item_matrix(df_interactions_train):
    user_item = df_interactions_train.pivot_table(
        index='user_id', 
        columns='item_id', 
        values='completed',  # Используем completed как меру взаимодействия
        fill_value=0
    )
    return user_item

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def find_similar_users_cosine(random_user_id, user_item_matrix, n_similar):
    
    if random_user_id not in user_item_matrix.index:
        return []
    
    # Получаем вектор целевого пользователя
    target_user_vector = user_item_matrix.loc[random_user_id].values.reshape(1, -1)
    
    # Вычисляем косинусное сходство со всеми пользователями
    similarities = cosine_similarity(target_user_vector, user_item_matrix.values)
    
    # Преобразуем в DataFrame для удобства
    similarity_df = pd.DataFrame({
        'user_id': user_item_matrix.index,
        'similarity': similarities[0]
    })
    
    # Убираем самого пользователя и сортируем по убыванию сходства
    similarity_df = similarity_df[similarity_df['user_id'] != random_user_id]
    similarity_df = similarity_df.sort_values('similarity', ascending=False)
    
    # Возвращаем топ-N похожих пользователей
    return similarity_df.head(n_similar)['user_id'].tolist()

In [ ]:
user_item_matrix = create_user_item_matrix(df_interactions_train)
print(f"Размер матрицы пользователь-предмет: {user_item_matrix.shape}")

In [ ]:
cosine_similar_users = find_similar_users_cosine(random_user_id, user_item_matrix)
print(f"Косинусное сходство (топ-5): {cosine_similar_users}")

In [ ]:
ground_truth = (
    last_interactions[last_interactions['completed'] == 1]
    .groupby('user_id')['item_id'].apply(list).to_dict()
)

In [ ]:
# Релевантные пользователи
eval_users = list(ground_truth.keys())

In [ ]:
def average_precision_at_k(recommended_items: List[int], relevant_items: List[int], k: int) -> float:

    if not relevant_items or k == 0:
        return 0.0

    # Обрезаем рекомендации до k
    recommended = recommended_items[:k]
    
    num_relevant_at_k = 0.0
    sum_precisions = 0.0

    for i, item_id in enumerate(recommended):
        if item_id in relevant_items:
            num_relevant_at_k += 1.0
            # Precision at i+1
            precision_at_i = num_relevant_at_k / (i + 1.0)
            sum_precisions += precision_at_i

    # Делим на общее количество релевантных элементов, которые можно было бы вернуть
    # Важно: в знаменателе min(N, K)
    # N - общее количество релевантных элементов для пользователя.
    # Так как мы не знаем общее количество релевантных элементов в полном датасете, 
    # а работаем только с ground_truth, то используем len(relevant_items).
    return sum_precisions / min(len(relevant_items), k)


In [ ]:
def mean_average_precision_at_k(model: ALS, users: List[int], ground_truth: dict, k: int) -> float:
    """
    Расчет Mean Average Precision at K (MAP@K)
    
    Parameters:
    - model: Обученная модель ALS
    - users: Список user_id для оценки
    - ground_truth: Словарь {user_id: [relevant_item_ids]}
    - k: Отсечка
    """
    sum_ap = 0.0
    num_users = len(users)

    if num_users == 0:
        return 0.0

    for user_id in tqdm(users, desc=f"Вычисление MAP@{k}"):
        # Получаем рекомендации (только item_id)
        recommended_items, _ = model.recommend(user_id, n_recommendations=k)
        
        # Получаем истинно релевантные элементы
        relevant_items = ground_truth.get(user_id, [])
        
        # Считаем AP@K для пользователя
        ap_k = average_precision_at_k(
            recommended_items.tolist(), # Преобразуем numpy array в list
            relevant_items, 
            k
        )
        sum_ap += ap_k

    # Возвращаем среднее AP
    return sum_ap / num_users

In [ ]:
# 5. Расчет MAP@10
K = 10
map_at_k = mean_average_precision_at_k(model, eval_users, ground_truth, K)

print(f"\n--- Результаты ---")
print(f"Список пользователей для оценки (с completed=1): {eval_users}")
print(f"Ground Truth (item_id с completed=1): {ground_truth}")
print(f"Mean Average Precision at K={K} (MAP@{K}): {map_at_k:.4f}")

In [ ]:
if eval_users:
    sample_user_id = eval_users[0]
    top_items, scores = model.recommend(sample_user_id, n_recommendations=K)
    print(f"\nТоп-{K} рекомендаций для пользователя {sample_user_id}:")
    for i, (item_id, score) in enumerate(zip(top_items, scores)):
        relevance_status = "✅ Релевантный" if item_id in ground_truth.get(sample_user_id, []) else "❌ Нерелевантный"
        print(f"  {i+1}. Item ID: {item_id}, Score: {score:.4f} ({relevance_status})")

In [ ]:
als_model_implicit = AlternatingLeastSquares(
    factors=50, 
    regularization=0.01, 
    iterations=15, 
    alpha=40, # Соответствует вашему alpha=40
    random_state=42 # Для воспроизводимости
)

In [ ]:
item_user_matrix = matrix.T.tocsr()

print("Начинаем обучение implicit.als (item x user matrix)...")
als_model_implicit.fit(item_user_matrix) 
print("Обучение implicit.als завершено.")

In [ ]:
# Загрузка данных (повтор для контекста)
# df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
# df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

# Создание колонки 'completed' (повтор для контекста)
# df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)

# 1. Получение уникальных ID и создание маппингов
# Для создания корректной разреженной матрицы (csr_matrix) важно, чтобы ID
# начинались с 0 и были последовательными.

# Создаем маппинг для user_id
unique_users = df_interactions['user_id'].unique()
user_to_index = {user_id: index for index, user_id in enumerate(unique_users)}
index_to_user = {index: user_id for user_id, index in user_to_index.items()}

# Создаем маппинг для item_id
unique_items = df_interactions['item_id'].unique()
item_to_index = {item_id: index for index, item_id in enumerate(unique_items)}
index_to_item = {index: item_id for item_id, index in item_to_index.items()}

# 2. Применение маппингов для получения индексов строк и столбцов
# Преобразуем исходные user_id и item_id в порядковые индексы
df_interactions['user_index'] = df_interactions['user_id'].map(user_to_index)
df_interactions['item_index'] = df_interactions['item_id'].map(item_to_index)

# 3. Подготовка данных для создания разреженной матрицы
# 'data' - значения (completed)
data = df_interactions['completed'].values
# 'rows' - индексы пользователей
rows = df_interactions['user_index'].values
# 'cols' - индексы элементов
cols = df_interactions['item_index'].values

# 4. Создание разреженной матрицы взаимодействий
# Shape (количество уникальных пользователей, количество уникальных элементов)
num_users = len(unique_users)
num_items = len(unique_items)

# Создаем матрицу
full_interaction_matrix = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))

print("--- Информация о полной матрице взаимодействий ---")
print(f"Общее количество уникальных пользователей: {num_users}")
print(f"Общее количество уникальных элементов (фильмов): {num_items}")
print(f"Количество ненулевых взаимодействий: {full_interaction_matrix.nnz}")
print(f"Размерность матрицы: {full_interaction_matrix.shape}")
print(f"Матрица взаимодействий (CSR формат):\n{full_interaction_matrix}")

# Сохранение маппингов для дальнейшего использования
# Теперь можно использовать full_interaction_matrix в модели ALS
# и user_to_index/index_to_user для преобразования ID.
# ALS_model.fit(full_interaction_matrix) 

In [ ]:
# Получение факторных матриц
U_matrix = model.user_factors
V_matrix = model.item_factors

# Расчет матриц сходства
# UserKNN Matrix (Приближение на основе латентных факторов)
UserKNN_Similarity_Matrix_Approx = np.dot(U_matrix, U_matrix.T)

# ItemKNN Matrix (Приближение на основе латентных факторов)
ItemKNN_Similarity_Matrix_Approx = np.dot(V_matrix, V_matrix.T)

print(f"\nРазмер Пользовательской факторной матрицы (U): {U_matrix.shape}")
print(f"Размер Предметной факторной матрицы (V): {V_matrix.shape}")
print(f"Размер Матрицы сходства UserKNN (U * U.T): {UserKNN_Similarity_Matrix_Approx.shape}")
print(f"Размер Матрицы сходства ItemKNN (V * V.T): {ItemKNN_Similarity_Matrix_Approx.shape}")

In [ ]:
def apk(actual, predicted, k=10):
    """
    Вычисляет Average Precision at K (AP@K).
    actual: список релевантных item_index (в тестовой выборке).
    predicted: отсортированный список рекомендованных item_index.
    """
    if not actual:
        return 0.0

    sum_precision = 0.0
    num_hits = 0.0
    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        # Если рекомендация p является релевантной
        if p in actual:
            num_hits += 1.0
            # Precision на текущем шаге
            precision_at_i = num_hits / (i + 1.0)
            sum_precision += precision_at_i

    # Делим на min(количество релевантных элементов, K)
    return sum_precision / min(len(actual), k)


def mapk(actuals, predicteds, k=10):
    """
    Вычисляет Mean Average Precision at K (MAP@K).
    actuals: список списков релевантных item_index.
    predicteds: список списков рекомендованных item_index.
    """
    # Игнорируем пользователей, у которых нет релевантных элементов в тесте (len(a) > 0)
    return np.mean([apk(a, p, k) for a, p in zip(actuals, predicteds) if len(a) > 0])


# --- Вычисление MAP@10 ---

# 1. Извлечение релевантных элементов (Actuals) из тестовой выборки
actuals = []
# Находим индексы пользователей, у которых есть взаимодействия в R_test
test_users = np.unique(R_test.nonzero()[0]) 

for user_idx in tqdm(test_users, desc="Извлечение релевантных элементов"):
    # Получаем item_indices, где R_test[user_idx, item_idx] > 0
    relevant_item_indices = R_test.indices[R_test.indptr[user_idx]:R_test.indptr[user_idx+1]]
    actuals.append(relevant_item_indices.tolist())

# 2. Получение рекомендаций (Predicteds) от модели ALS
predicteds = []
for user_idx in tqdm(test_users, desc="Получение рекомендаций"):
    # Получаем N=10 рекомендаций для пользователя
    # model.get_recommendations возвращает item_indices, которые еще не просмотрены
    recommendations, _ = model.get_recommendations(user_idx, R_train, N=K_REC)
    predicteds.append(recommendations.tolist())
    
# 3. Вычисление MAP@10
map_at_10 = mapk(actuals, predicteds, k=K_REC)

print(f"\n--- Результат оценки ---")
print(f"MAP@{K_REC} = {map_at_10:.4f}")

In [ ]:
selected_users = np.random.choice(df_interactions['user_id'].unique(), size=10, replace=False)
df_sampled = df_interactions[df_interactions['user_id'].isin(selected_users)].copy()

In [ ]:
def create_interaction_matrix(df, user_map, item_map, shape):
    """Создает разреженную матрицу (CSR) на основе маппинга."""
    df_temp = df.copy()
    df_temp['user_index'] = df_temp['user_id'].map(user_map)
    df_temp['item_index'] = df_temp['item_id'].map(item_map)
    
    # Удаляем строки, где маппинг не сработал (должны быть в исходном наборе)
    df_temp.dropna(subset=['user_index', 'item_index'], inplace=True)
    
    R = csr_matrix((df_temp['completed'].astype(np.float32), 
                    (df_temp['user_index'].astype(int), df_temp['item_index'].astype(int))), 
                   shape=shape)
    return R

In [ ]:
user_map = {user_id: index for index, user_id in enumerate(df_sampled['user_id'].unique())}
item_map = {item_id: index for index, item_id in enumerate(df_sampled['item_id'].unique())}
N_USERS = len(user_map)
N_ITEMS = len(item_map)
MATRIX_SHAPE = (N_USERS, N_ITEMS)

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- Класс ALS (обертка вокруг implicit.als) ---
class ALS:
    """
    Пользовательский класс ALS, использующий implicit.als для обучения,
    чтобы соответствовать импортам и структуре вашего кода.
    """
    def __init__(self, n_factors, regularization, iterations, random_state=42):
        self.n_factors = n_factors
        self.regularization = regularization
        self.iterations = iterations
        self.loss_history = []
        
        # Инициализация модели implicit.als
        self.model = AlternatingLeastSquares(
            factors=n_factors, 
            regularization=regularization, 
            iterations=iterations, 
            random_state=random_state
        )
        self.user_factors = None
        self.item_factors = None

    def fit(self, user_item_matrix):
        """Обучение модели на разреженной матрице взаимодействий R (Users x Items)."""
        print("Начало обучения ALS...")
        # implicit.als ожидает матрицу Items x Users для implicit_cf
        item_user_matrix = user_item_matrix.T.tocsr() 
        self.model.fit(item_user_matrix, show_progress=True)
        
        # Сохраняем полученные факторные матрицы
        self.item_factors = self.model.item_factors # V (N_items x N_factors)
        self.user_factors = self.model.user_factors # U (N_users x N_factors)
        print("Обучение ALS завершено.")
        
    def get_recommendations(self, user_index, user_item_matrix, N=10):
        """Получение N рекомендаций для заданного пользователя."""
        recommendations = self.model.recommend(
            user_index, 
            user_item_matrix[user_index], 
            N=N, 
            filter_already_liked_items=True 
        )
        # recommendations: (item_indices, scores)
        return recommendations
    
# --- Функции подготовки данных ---
def create_interaction_matrix(df, user_map, item_map, shape):
    """Создает разреженную матрицу (CSR) на основе маппинга."""
    df_temp = df.copy()
    df_temp['user_index'] = df_temp['user_id'].map(user_map)
    df_temp['item_index'] = df_temp['item_id'].map(item_map)
    
    # Удаляем строки, где маппинг не сработал (должны быть в исходном наборе)
    df_temp.dropna(subset=['user_index', 'item_index'], inplace=True)
    
    R = csr_matrix((df_temp['completed'].astype(np.float32), 
                    (df_temp['user_index'].astype(int), df_temp['item_index'].astype(int))), 
                   shape=shape)
    return R

# --- Основной скрипт ---

# 1. Загрузка и предобработка данных (как в вашем коде)
# Внимание: замените 'data_original/data_original/interactions.csv' на свой путь, 
# если он отличается.
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)

# Для демонстрации и предотвращения ошибок памяти, используем 1000 случайных пользователей
selected_users = np.random.choice(df_interactions['user_id'].unique(), size=1000, replace=False)
df_sampled = df_interactions[df_interactions['user_id'].isin(selected_users)].copy()

# Создание общего маппинга для всей выборки
user_map = {user_id: index for index, user_id in enumerate(df_sampled['user_id'].unique())}
item_map = {item_id: index for index, item_id in enumerate(df_sampled['item_id'].unique())}
N_USERS = len(user_map)
N_ITEMS = len(item_map)
MATRIX_SHAPE = (N_USERS, N_ITEMS)

# 2. Разделение на обучающую и тестовую выборки
train_df, test_df = train_test_split(df_sampled, test_size=0.2, random_state=42)

R_train = create_interaction_matrix(train_df, user_map, item_map, MATRIX_SHAPE)
R_test = create_interaction_matrix(test_df, user_map, item_map, MATRIX_SHAPE)

# 3. Инициализация и обучение модели
N_FACTORS = 50
REGULARIZATION = 0.01
ITERATIONS = 15
K_REC = 10 

model = ALS(n_factors=N_FACTORS, regularization=REGULARIZATION, iterations=ITERATIONS)
model.fit(R_train)